# 📘 Pydantic for Data Engineering
## Databricks Notebook — Schema Validation for Pipelines

**What you'll learn:**
- Pydantic v2 for data validation at pipeline boundaries
- Field validation, nested models, custom types
- Batch validation with error collection
- Quarantine pattern (valid → lake, invalid → quarantine)
- Configuration management
- API contract validation

**Prerequisites:** Notebooks 01-10 (techmart_dw)

---

In [0]:
%pip install pydantic --quiet

In [0]:
# Verify installation
import pydantic
print(f"Pydantic version: {pydantic.__version__}")
assert pydantic.__version__.startswith("2"), "Need Pydantic v2!"
print("✅ Pydantic v2 ready")

---
# 📗 Section 1: Why Pydantic for Data Engineering

**Problem:** Raw data from APIs, files, and streams is messy and untyped.
Without validation, bad data silently corrupts your data lake.

**Pydantic solves this by:**
1. Defining strict schemas (what data SHOULD look like)
2. Validating on instantiation (catches errors immediately)
3. Type coercion (string "123" → int 123 automatically)
4. Detailed error messages (which field, what's wrong, what was expected)

**Where to use in a pipeline:**
```
Source → [VALIDATE HERE] → Bronze → [VALIDATE HERE] → Silver → Gold
         (API contracts)         (business rules)
```

---
# 📗 Section 2: Pydantic Basics

In [0]:
from pydantic import BaseModel, Field, ConfigDict
from typing import Optional
from datetime import date, datetime

# Define a schema
class Customer(BaseModel):
    customer_id: int
    first_name: str
    last_name: str
    email: Optional[str] = None
    lifetime_value: float = 0.0
    is_active: bool = True
    registration_date: date

# Valid data — works perfectly
customer = Customer(
    customer_id=1,
    first_name="John",
    last_name="Doe",
    email="john@example.com",
    lifetime_value=15000.50,
    registration_date="2023-01-15"  # String auto-coerced to date!
)
print(f"Created: {customer}")
print(f"Type of registration_date: {type(customer.registration_date)}")

In [0]:
# Type coercion: Pydantic converts compatible types automatically
customer2 = Customer(
    customer_id="42",           # str → int (coerced!)
    first_name="Jane",
    last_name="Smith",
    lifetime_value="25000.75",  # str → float (coerced!)
    is_active="true",           # str → bool (coerced!)
    registration_date="2024-06-01"
)
print(f"customer_id type: {type(customer2.customer_id)} = {customer2.customer_id}")
print(f"lifetime_value type: {type(customer2.lifetime_value)} = {customer2.lifetime_value}")

In [0]:
# ValidationError: what happens with BAD data
from pydantic import ValidationError

try:
    bad_customer = Customer(
        customer_id="not_a_number",  # Can't coerce to int
        first_name=None,              # Required field
        last_name="Test",
        registration_date="invalid"   # Can't parse as date
    )
except ValidationError as e:
    print(f"Validation failed with {e.error_count()} errors:")
    for err in e.errors():
        print(f"  ❌ {err['loc'][0]}: {err['msg']} (type: {err['type']})")

In [0]:
# Field aliases (map external names to internal names)
class ApiCustomer(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    
    customer_id: int = Field(alias="customerId")
    first_name: str = Field(alias="firstName")
    last_name: str = Field(alias="lastName")
    email: Optional[str] = Field(default=None, alias="emailAddress")

# Works with either alias or field name
from_api = ApiCustomer(customerId=1, firstName="John", lastName="Doe", emailAddress="j@test.com")
print(f"From API: {from_api}")
print(f"As dict (by alias): {from_api.model_dump(by_alias=True)}")

---
# 📗 Section 3: Field Validation

In [0]:
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import Optional, Literal
from datetime import date

class Order(BaseModel):
    order_id: int = Field(gt=0)
    customer_id: int = Field(gt=0)
    total_amount: float = Field(ge=0, description="Must be non-negative")
    status: Literal["pending", "completed", "cancelled", "shipped", "returned"]
    order_date: date
    discount_amount: float = Field(ge=0, le=10000, default=0)
    
    # Custom field validator
    @field_validator("status", mode="before")
    @classmethod
    def normalize_status(cls, v):
        if isinstance(v, str):
            return v.strip().lower()
        return v
    
    # Cross-field validation
    @model_validator(mode="after")
    def check_discount_not_exceeds_total(self):
        if self.discount_amount > self.total_amount:
            raise ValueError("discount_amount cannot exceed total_amount")
        return self

# Valid order
order = Order(
    order_id=1, customer_id=100, total_amount=500.0,
    status="  COMPLETED  ",  # Gets normalized!
    order_date="2024-06-01", discount_amount=50.0
)
print(f"Order: {order}")
print(f"Status (normalized): '{order.status}'")

In [0]:
# Validation failures
from pydantic import ValidationError

test_cases = [
    {"order_id": -1, "customer_id": 1, "total_amount": 100, "status": "completed", "order_date": "2024-01-01"},
    {"order_id": 1, "customer_id": 1, "total_amount": 100, "status": "INVALID_STATUS", "order_date": "2024-01-01"},
    {"order_id": 1, "customer_id": 1, "total_amount": 50, "status": "completed", "order_date": "2024-01-01", "discount_amount": 100},
]

for i, data in enumerate(test_cases, 1):
    try:
        Order(**data)
        print(f"  Case {i}: ✅ Valid")
    except ValidationError as e:
        print(f"  Case {i}: ❌ {e.errors()[0]['msg']}")

In [0]:
# ============================================
# 🎯 YOUR TURN: Validate Payment Records
# ============================================
# Define a Payment model with:
# - payment_id: int (> 0)
# - order_id: int (> 0)
# - amount: float (> 0)
# - payment_method: Literal["credit_card", "debit_card", "paypal", "bank_transfer"]
# - payment_date: date
# - status: Literal["completed", "pending", "failed", "refunded"]
# - @field_validator: normalize payment_method (strip, lowercase)
# - @model_validator: if status is "refunded", amount must be negative (or allow it)
#
# Test with 3 valid and 2 invalid records
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
class Payment(BaseModel):
    payment_id: int = Field(gt=0)
    order_id: int = Field(gt=0)
    amount: float = Field(gt=0)
    payment_method: Literal["credit_card", "debit_card", "paypal", "bank_transfer"]
    payment_date: date
    status: Literal["completed", "pending", "failed", "refunded"]
    
    @field_validator("payment_method", mode="before")
    @classmethod
    def normalize_method(cls, v):
        return v.strip().lower() if isinstance(v, str) else v

# Test
valid = [
    {"payment_id": 1, "order_id": 10, "amount": 150.0, "payment_method": "CREDIT_CARD", "payment_date": "2024-01-01", "status": "completed"},
    {"payment_id": 2, "order_id": 11, "amount": 200.0, "payment_method": "paypal", "payment_date": "2024-01-02", "status": "pending"},
]
invalid = [
    {"payment_id": -1, "order_id": 10, "amount": 150.0, "payment_method": "credit_card", "payment_date": "2024-01-01", "status": "completed"},
    {"payment_id": 3, "order_id": 10, "amount": 150.0, "payment_method": "bitcoin", "payment_date": "2024-01-01", "status": "completed"},
]

for data in valid:
    p = Payment(**data)
    print(f"  ✅ Payment {p.payment_id}: {p.payment_method} ${p.amount}")

for data in invalid:
    try:
        Payment(**data)
    except ValidationError as e:
        print(f"  ❌ {e.errors()[0]['loc'][0]}: {e.errors()[0]['msg']}")

---
# 📗 Section 4: Nested Models

In [0]:
from pydantic import BaseModel, Field
from typing import List, Optional
from datetime import datetime

class Address(BaseModel):
    street: str
    city: str
    state: str = Field(min_length=2, max_length=2)
    zip_code: str = Field(pattern=r"^\d{5}$")

class OrderItem(BaseModel):
    product_id: int
    quantity: int = Field(gt=0)
    unit_price: float = Field(gt=0)
    
    @property
    def line_total(self) -> float:
        return self.quantity * self.unit_price

class FullOrder(BaseModel):
    order_id: int
    customer_name: str
    shipping_address: Address  # Nested model
    items: List[OrderItem]     # List of nested models
    order_date: datetime
    
    @property
    def total(self) -> float:
        return sum(item.line_total for item in self.items)

# Validate complex nested data (like an API response)
order_data = {
    "order_id": 1001,
    "customer_name": "John Doe",
    "shipping_address": {"street": "123 Main St", "city": "Austin", "state": "TX", "zip_code": "78701"},
    "items": [
        {"product_id": 1, "quantity": 2, "unit_price": 29.99},
        {"product_id": 5, "quantity": 1, "unit_price": 149.99},
    ],
    "order_date": "2024-06-15T10:30:00"
}

order = FullOrder(**order_data)
print(f"Order {order.order_id}: {len(order.items)} items, total=${order.total:.2f}")
print(f"Ship to: {order.shipping_address.city}, {order.shipping_address.state}")

---
# 📗 Section 5: Data Pipeline Integration

## Batch Validation with Quarantine Pattern

In [0]:
from pydantic import BaseModel, Field, ValidationError, field_validator
from typing import Literal, Optional
from datetime import date

class SilverOrder(BaseModel):
    """Schema for Silver layer orders."""
    order_id: int = Field(gt=0)
    customer_id: int = Field(gt=0)
    total_amount: float = Field(ge=0)
    status: Literal["completed", "shipped", "processing", "pending", "cancelled", "returned"]
    order_date: date
    payment_method: str = Field(min_length=1)
    
    @field_validator("status", mode="before")
    @classmethod
    def normalize_status(cls, v):
        return v.strip().lower() if isinstance(v, str) else v
    
    @field_validator("payment_method", mode="before")
    @classmethod
    def normalize_payment(cls, v):
        return v.strip().lower() if isinstance(v, str) else v

def validate_batch(records: list, model_class) -> tuple:
    """Validate a batch of records, separating valid from invalid.
    
    Returns: (valid_records, error_records)
    """
    valid = []
    errors = []
    
    for i, record in enumerate(records):
        try:
            validated = model_class(**record)
            valid.append(validated.model_dump())
        except ValidationError as e:
            errors.append({
                "record_index": i,
                "record": record,
                "error_count": e.error_count(),
                "errors": [{"field": err["loc"][0], "message": err["msg"], "type": err["type"]} for err in e.errors()]
            })
    
    return valid, errors

In [0]:
# Load real data and validate
import pandas as pd

orders_pdf = spark.table("techmart_dw.orders").limit(1000).toPandas()
records = orders_pdf.to_dict("records")

# Validate batch
valid, errors = validate_batch(records, SilverOrder)

print(f"Batch validation results:")
print(f"  Total records: {len(records):,}")
print(f"  Valid: {len(valid):,} ({len(valid)/len(records)*100:.1f}%)")
print(f"  Invalid: {len(errors):,} ({len(errors)/len(records)*100:.1f}%)")

if errors:
    print(f"\nSample errors:")
    for err in errors[:3]:
        print(f"  Record {err['record_index']}: {err['errors'][0]['field']} — {err['errors'][0]['message']}")

In [0]:
# Write valid to Silver, invalid to Quarantine
valid_df = spark.createDataFrame(pd.DataFrame(valid))
valid_df.write.format("delta").mode("overwrite").saveAsTable("techmart_dw.silver_orders_validated")
print(f"✅ Valid → silver_orders_validated: {valid_df.count():,} rows")

if errors:
    # Flatten errors for quarantine table
    quarantine_records = []
    for err in errors:
        quarantine_records.append({
            "record_index": err["record_index"],
            "error_count": err["error_count"],
            "first_error_field": err["errors"][0]["field"],
            "first_error_msg": err["errors"][0]["message"],
        })
    quarantine_df = spark.createDataFrame(pd.DataFrame(quarantine_records))
    quarantine_df.write.format("delta").mode("overwrite").saveAsTable("techmart_dw.quarantine_orders")
    print(f"⚠️ Invalid → quarantine_orders: {quarantine_df.count():,} rows")

---
# 📗 Section 6: Configuration Management

In [0]:
from pydantic import BaseModel, Field
from typing import Optional, List

class DatabaseConfig(BaseModel):
    host: str
    port: int = Field(ge=1, le=65535, default=5432)
    database: str
    username: str
    password: str = Field(min_length=8)

class PipelineConfig(BaseModel):
    name: str
    version: str = Field(pattern=r"^\d+\.\d+\.\d+$")
    source: DatabaseConfig
    target_table: str
    batch_size: int = Field(ge=100, le=1000000, default=10000)
    retry_count: int = Field(ge=0, le=10, default=3)
    quality_threshold: float = Field(ge=0.0, le=1.0, default=0.95)
    enabled: bool = True

# Validate a pipeline config
config = PipelineConfig(
    name="orders_pipeline",
    version="2.1.0",
    source=DatabaseConfig(
        host="db.example.com", port=5432,
        database="production", username="etl_user", password="secure_pass_123"
    ),
    target_table="silver_orders",
    batch_size=50000,
    quality_threshold=0.98
)
print(f"Pipeline: {config.name} v{config.version}")
print(f"Source: {config.source.host}:{config.source.port}/{config.source.database}")
print(f"Batch size: {config.batch_size:,}")

---
# 📗 Section 7: API Contract Validation

In [0]:
from pydantic import BaseModel, Field
from typing import List, Optional
from datetime import datetime

# Define API contract
class WebhookPayload(BaseModel):
    event_type: str = Field(pattern=r"^(order|payment|shipment)\.(created|updated|deleted)$")
    timestamp: datetime
    source: str
    data: dict
    retry_count: int = Field(ge=0, default=0)

# Validate incoming webhooks
test_webhooks = [
    {"event_type": "order.created", "timestamp": "2024-06-15T10:30:00Z", "source": "shopify", "data": {"order_id": 123}},
    {"event_type": "payment.updated", "timestamp": "2024-06-15T10:31:00Z", "source": "stripe", "data": {"payment_id": 456}},
    {"event_type": "invalid.event", "timestamp": "2024-06-15T10:32:00Z", "source": "unknown", "data": {}},
]

for webhook in test_webhooks:
    try:
        validated = WebhookPayload(**webhook)
        print(f"  ✅ {validated.event_type} from {validated.source}")
    except ValidationError as e:
        print(f"  ❌ {webhook['event_type']}: {e.errors()[0]['msg']}")

---
# 🚀 Section 8: Mini Projects

## Project 1: Complete Ingestion Validator

In [0]:
from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import Literal, Optional
from datetime import date, datetime
import pandas as pd

class CustomerSchema(BaseModel):
    """Validation schema for customer records."""
    customer_id: int = Field(gt=0)
    first_name: str = Field(min_length=1)
    last_name: str = Field(min_length=1)
    email: Optional[str] = None
    customer_segment: Literal["Premium", "Standard", "Basic", "New"]
    lifetime_value: float = Field(ge=0)
    is_active: bool
    
    @field_validator("first_name", "last_name", mode="before")
    @classmethod
    def clean_name(cls, v):
        return v.strip().title() if isinstance(v, str) else v
    
    @field_validator("email", mode="before")
    @classmethod
    def validate_email(cls, v):
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return None
        if isinstance(v, str) and "@" not in v:
            raise ValueError("Invalid email format")
        return v.strip().lower() if isinstance(v, str) else v

# Load and validate
customers_pdf = spark.table("techmart_dw.customers").limit(500).toPandas()
records = customers_pdf.to_dict("records")

valid, errors = validate_batch(records, CustomerSchema)

print(f"Customer Validation:")
print(f"  Total: {len(records)}")
print(f"  Valid: {len(valid)} ({len(valid)/len(records)*100:.1f}%)")
print(f"  Invalid: {len(errors)} ({len(errors)/len(records)*100:.1f}%)")

# Error breakdown
if errors:
    error_fields = {}
    for err in errors:
        for e in err["errors"]:
            field = e["field"]
            error_fields[field] = error_fields.get(field, 0) + 1
    print(f"\n  Errors by field:")
    for field, count in sorted(error_fields.items(), key=lambda x: -x[1]):
        print(f"    {field}: {count}")

## Project 2: Schema Registry Simulator

In [0]:
from pydantic import BaseModel, Field
from typing import Optional, Literal
from datetime import date

# Schema v1
class CustomerV1(BaseModel):
    customer_id: int
    name: str
    email: str
    segment: str

# Schema v2 (added fields, renamed field)
class CustomerV2(BaseModel):
    customer_id: int
    first_name: str
    last_name: str
    email: Optional[str] = None
    segment: Literal["Premium", "Standard", "Basic", "New"]
    lifetime_value: float = 0.0  # New field with default
    registration_date: Optional[date] = None  # New field

# Version router
def validate_record(record: dict, version: int):
    schema_map = {1: CustomerV1, 2: CustomerV2}
    model = schema_map.get(version)
    if not model:
        raise ValueError(f"Unknown schema version: {version}")
    return model(**record)

# Test backward compatibility
v1_record = {"customer_id": 1, "name": "John Doe", "email": "john@test.com", "segment": "Premium"}
v2_record = {"customer_id": 2, "first_name": "Jane", "last_name": "Smith", "email": "jane@test.com", "segment": "Standard", "lifetime_value": 5000.0}

print(f"V1 record: {validate_record(v1_record, 1)}")
print(f"V2 record: {validate_record(v2_record, 2)}")

---
# 🏆 Section 9: Interview Questions

## Q1: How do you validate data at pipeline boundaries?

Use Pydantic models at ingestion points:
1. Define a schema (BaseModel) for expected data shape
2. Validate each record on entry (`model_validate(record)`)
3. Separate valid from invalid (quarantine pattern)
4. Log validation errors for monitoring
5. Only valid records proceed to the next layer

## Q2: Schema-on-read vs schema-on-write?

- **Schema-on-read (Bronze):** Accept anything, validate later. Flexible but risky.
- **Schema-on-write (Silver):** Validate BEFORE writing. Strict but safe.
- **Best practice:** Schema-on-read for Bronze, schema-on-write for Silver/Gold.

## Q3: Schema evolution with validation?

1. Version your Pydantic models (CustomerV1, CustomerV2)
2. New fields get defaults (`Optional[str] = None`)
3. Route records to correct version based on metadata
4. Old records still validate against new schema (backward compatible)
5. Breaking changes require migration scripts

## Q4: Quarantine pattern?

1. Validate batch of records
2. Valid → write to Silver table
3. Invalid → write to quarantine table (with error details)
4. Monitor quarantine growth (alerts if > threshold)
5. Periodically review and fix quarantine records

## Q5: Pydantic v1 vs v2?

| v1 | v2 |
|---|---|
| `@validator` | `@field_validator` |
| `@root_validator` | `@model_validator` |
| `.dict()` | `.model_dump()` |
| `.json()` | `.model_dump_json()` |
| `class Config` | `model_config = ConfigDict(...)` |
| `parse_obj()` | `model_validate()` |

## Q6: Collect ALL errors in a batch?

Don't fail on first error. Loop through records, catch `ValidationError` per record,
collect all errors into a list. Report aggregate stats (errors per field, per type).

## Q7: Pydantic vs Great Expectations vs PySpark schema?

- **Pydantic:** Row-level validation, Python objects, API contracts, configs
- **Great Expectations:** Column-level expectations, statistical checks, data profiling
- **PySpark schema:** Type enforcement on read, no business rule validation

Use Pydantic for record-level, GE for dataset-level, PySpark for type safety.

## Q8: Discriminated unions?

Use `Literal` type field as discriminator to pick the right model:
```python
class CardPayment(BaseModel):
    type: Literal["card"]
    card_number: str

class BankPayment(BaseModel):
    type: Literal["bank"]
    account_number: str

Payment = CardPayment | BankPayment  # Python picks based on 'type' field
```

---
# 📗 Section 4: Advanced Validators & Custom Types

Pydantic v2 validators let you enforce business rules beyond simple type
checking — essential for DE pipelines where data quality is critical.

In [0]:
from pydantic import BaseModel, Field, field_validator, model_validator
from pydantic import ValidationError
from typing import Optional, List, Literal
from decimal import Decimal
from datetime import datetime
import re

# --- Custom validators ---
class CustomerRecord(BaseModel):
    customer_id: int = Field(gt=0, description="Must be positive")
    email: str
    phone: Optional[str] = None
    age: int = Field(ge=0, le=150)
    annual_income: Decimal = Field(ge=0, decimal_places=2)
    tier: Literal["bronze", "silver", "gold", "platinum"]
    
    @field_validator("email")
    @classmethod
    def validate_email(cls, v):
        pattern = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"
        if not re.match(pattern, v):
            raise ValueError(f"Invalid email format: {v}")
        return v.lower().strip()
    
    @field_validator("phone")
    @classmethod
    def validate_phone(cls, v):
        if v is None:
            return v
        # Remove formatting
        cleaned = re.sub(r"[\s\-\(\)\+]", "", v)
        if not cleaned.isdigit() or len(cleaned) < 10:
            raise ValueError(f"Invalid phone: {v}")
        return cleaned
    
    @model_validator(mode="after")
    def validate_tier_income(self):
        """Platinum customers must have income > 100k."""
        if self.tier == "platinum" and self.annual_income < 100_000:
            raise ValueError("Platinum tier requires annual_income >= 100,000")
        return self

# Test valid record
valid = CustomerRecord(
    customer_id=1, email="  John.Doe@Example.COM  ",
    phone="(555) 123-4567", age=35,
    annual_income="150000.00", tier="platinum"
)
print("Valid record:", valid.model_dump())

# Test invalid records
test_cases = [
    {"customer_id": -1, "email": "bad", "age": 35, "annual_income": "100", "tier": "gold"},
    {"customer_id": 2, "email": "ok@test.com", "age": 35, "annual_income": "50000", "tier": "platinum"},
]
for case in test_cases:
    try:
        CustomerRecord(**case)
    except ValidationError as e:
        print(f"\nValidation failed: {[err['msg'] for err in e.errors()]}")

In [0]:
# --- Nested models for complex DE schemas ---
class Address(BaseModel):
    street: str = Field(min_length=1)
    city: str = Field(min_length=1)
    country: str = Field(min_length=2, max_length=2)  # ISO 2-letter code
    postal_code: str

class OrderItem(BaseModel):
    product_id: int = Field(gt=0)
    quantity: int = Field(gt=0, le=1000)
    unit_price: Decimal = Field(gt=0, decimal_places=2)
    
    @property
    def line_total(self) -> Decimal:
        return self.quantity * self.unit_price

class Order(BaseModel):
    order_id: int = Field(gt=0)
    customer_id: int = Field(gt=0)
    order_date: datetime
    shipping_address: Address
    items: List[OrderItem] = Field(min_length=1)
    status: Literal["pending", "processing", "shipped", "delivered", "cancelled"]
    
    @model_validator(mode="after")
    def validate_items_not_empty(self):
        if not self.items:
            raise ValueError("Order must have at least one item")
        return self
    
    @property
    def total_amount(self) -> Decimal:
        return sum(item.line_total for item in self.items)

# Create a valid order
order = Order(
    order_id=1001,
    customer_id=42,
    order_date="2024-01-15T10:30:00",
    shipping_address={"street": "123 Main St", "city": "Seattle", "country": "US", "postal_code": "98101"},
    items=[
        {"product_id": 1, "quantity": 2, "unit_price": "29.99"},
        {"product_id": 2, "quantity": 1, "unit_price": "149.99"},
    ],
    status="pending"
)
print(f"Order {order.order_id}: ${order.total_amount}")
print(f"Items: {len(order.items)}")
print(f"Shipping to: {order.shipping_address.city}, {order.shipping_address.country}")

## 4.2 Batch Validation with Error Collection

In DE pipelines, you want to collect ALL errors, not stop at the first one.
Here's the production pattern for batch validation.

In [0]:
from typing import Tuple
import json

def validate_batch(records: list, model_class, source_name="batch") -> Tuple[list, list]:
    """
    Validate a batch of records against a Pydantic model.
    
    Returns:
        (valid_records, error_records)
        error_records include the original data + error details
    """
    valid = []
    errors = []
    
    for i, record in enumerate(records):
        try:
            validated = model_class(**record)
            valid.append(validated.model_dump())
        except ValidationError as e:
            error_entry = {
                "_source": source_name,
                "_row_index": i,
                "_raw_data": record,
                "_errors": [
                    {
                        "field": ".".join(str(x) for x in err["loc"]),
                        "message": err["msg"],
                        "type": err["type"],
                    }
                    for err in e.errors()
                ],
                "_error_count": len(e.errors()),
            }
            errors.append(error_entry)
    
    total = len(records)
    print(f"\n📊 Validation Summary: {source_name}")
    print(f"   Total:   {total}")
    print(f"   Valid:   {len(valid)} ({len(valid)/total*100:.1f}%)")
    print(f"   Invalid: {len(errors)} ({len(errors)/total*100:.1f}%)")
    
    return valid, errors

# Test batch validation
class ProductRecord(BaseModel):
    product_id: int = Field(gt=0)
    name: str = Field(min_length=1, max_length=100)
    price: float = Field(gt=0)
    category: Literal["electronics", "clothing", "food", "books"]

raw_products = [
    {"product_id": 1, "name": "Laptop", "price": 999.99, "category": "electronics"},
    {"product_id": -1, "name": "Bad ID", "price": 50.0, "category": "books"},
    {"product_id": 2, "name": "T-Shirt", "price": 29.99, "category": "clothing"},
    {"product_id": 3, "name": "", "price": 10.0, "category": "food"},
    {"product_id": 4, "name": "Novel", "price": -5.0, "category": "books"},
    {"product_id": 5, "name": "Phone", "price": 699.0, "category": "gadgets"},  # invalid category
]

valid_products, error_products = validate_batch(raw_products, ProductRecord, "products_api")

print("\nValid products:")
for p in valid_products:
    print(f"  {p}")

print("\nError records:")
for e in error_products:
    print(f"  Row {e['_row_index']}: {[err['message'] for err in e['_errors']]}")

---
# 📗 Section 5: Pydantic Settings & Config Management

Pydantic Settings is the standard way to manage environment-based
configuration in DE pipelines.

In [0]:
from pydantic_settings import BaseSettings
from pydantic import SecretStr
from typing import Optional
import os

# Simulate environment variables
os.environ["DB_HOST"] = "prod-db.company.com"
os.environ["DB_PORT"] = "5432"
os.environ["DB_NAME"] = "datawarehouse"
os.environ["DB_PASSWORD"] = "super_secret_123"
os.environ["ENVIRONMENT"] = "production"
os.environ["MAX_RETRIES"] = "3"

class PipelineSettings(BaseSettings):
    """
    Pipeline configuration loaded from environment variables.
    Pydantic automatically reads from env vars matching field names.
    """
    # Database
    db_host: str
    db_port: int = 5432
    db_name: str
    db_password: SecretStr  # Never logged/printed
    db_pool_size: int = 5
    
    # Pipeline behavior
    environment: Literal["development", "staging", "production"] = "development"
    max_retries: int = Field(default=3, ge=1, le=10)
    batch_size: int = Field(default=1000, ge=100, le=100_000)
    
    # Optional
    slack_webhook_url: Optional[str] = None
    
    @property
    def db_url(self) -> str:
        """Build connection string (password hidden)."""
        return f"postgresql://{self.db_host}:{self.db_port}/{self.db_name}"
    
    @property
    def is_production(self) -> bool:
        return self.environment == "production"
    
    class Config:
        env_file = ".env"
        case_sensitive = False

settings = PipelineSettings()
print(f"Environment: {settings.environment}")
print(f"DB URL: {settings.db_url}")
print(f"Password (SecretStr): {settings.db_password}")  # Shows ****
print(f"Password (actual): {settings.db_password.get_secret_value()}")
print(f"Is production: {settings.is_production}")
print(f"Max retries: {settings.max_retries}")

---
# 📗 Section 6: Practice Exercises

In [0]:
# ============================================================
# EXERCISE 1: API Response Validator
# ============================================================
# Build a Pydantic model for a REST API response that:
# - Has a status field (success/error)
# - Has optional data (list of records) and error_message
# - Validates that data is present when status=success
# - Validates that error_message is present when status=error

from typing import Any, Dict

class APIResponse(BaseModel):
    status: Literal["success", "error"]
    data: Optional[List[Dict[str, Any]]] = None
    error_message: Optional[str] = None
    timestamp: datetime = Field(default_factory=datetime.utcnow)
    request_id: str
    
    @model_validator(mode="after")
    def validate_status_fields(self):
        if self.status == "success" and self.data is None:
            raise ValueError("data must be provided when status=success")
        if self.status == "error" and not self.error_message:
            raise ValueError("error_message must be provided when status=error")
        return self

# Test cases
test_responses = [
    {"status": "success", "data": [{"id": 1}], "request_id": "req-001"},
    {"status": "error", "error_message": "Not found", "request_id": "req-002"},
    {"status": "success", "request_id": "req-003"},  # Should fail
    {"status": "error", "request_id": "req-004"},    # Should fail
]

for resp in test_responses:
    try:
        r = APIResponse(**resp)
        print(f"✅ {resp['request_id']}: {r.status}")
    except ValidationError as e:
        print(f"❌ {resp['request_id']}: {[err['msg'] for err in e.errors()]}")

In [0]:
# ============================================================
# EXERCISE 2: Schema Evolution Handling
# ============================================================
# Handle backward-compatible schema changes using Pydantic

class OrderV1(BaseModel):
    """Original schema."""
    order_id: int
    amount: float
    status: str

class OrderV2(BaseModel):
    """New schema with additional fields (backward compatible)."""
    order_id: int
    amount: float
    status: str
    # New fields with defaults (backward compatible)
    currency: str = "USD"
    tax_amount: float = 0.0
    discount_pct: float = Field(default=0.0, ge=0, le=100)
    
    @field_validator("status")
    @classmethod
    def normalize_status(cls, v):
        return v.lower().strip()
    
    @property
    def total_with_tax(self) -> float:
        return self.amount + self.tax_amount

# Old records (V1 format) can be loaded into V2 model
old_records = [
    {"order_id": 1, "amount": 100.0, "status": "COMPLETED"},
    {"order_id": 2, "amount": 200.0, "status": "  Pending  "},
]

print("Loading V1 records into V2 schema:")
for rec in old_records:
    order = OrderV2(**rec)
    print(f"  Order {order.order_id}: status={order.status}, "
          f"currency={order.currency}, total={order.total_with_tax}")

---
# ✅ Notebook Complete!

**What was covered:**
- Pydantic v2 basics: BaseModel, Field, type coercion, ValidationError
- Field validation: @field_validator, @model_validator, constraints
- Nested models: Address inside Customer, List[OrderItem]
- Pipeline integration: batch validation, quarantine pattern
- Configuration management: validated pipeline configs
- API contract validation: webhook payloads
- 2 mini projects: Ingestion Validator, Schema Registry
- 8 interview questions

**Next:** Notebook 12 — Requests & APIs for Data Engineering

---